# Modelo 2: Match Predictor - Power EDA Edition
**Machine Learning I - Universidad Externado de Colombia**

Este notebook profundiza en la biomecánica del descanso, el impacto de los árbitros y la regresión penalizada para predecir el volumen de goles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from datetime import timedelta
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import kruskal

# Estilo Premier League
PL_PURPLE = '#38003c'
PL_CYAN = '#00ff85'
PL_PINK = '#e90052'
sns.set_theme(style="darkgrid")
plt.rcParams['figure.facecolor'] = 'white'

## 1. Power EDA: Cronología de la Temporada
Visualizamos cómo varía el volumen de goles a lo largo del tiempo para detectar patrones de desgaste.

In [ ]:
df_matches = pd.read_csv('../../data/matches.csv')
df_matches['date_parsed'] = pd.to_datetime(df_matches['date'], format='%d/%m/%Y')
df_matches = df_matches.sort_values('date_parsed')

plt.figure(figsize=(12, 5))
df_matches.groupby('date_parsed')['total_goals'].mean().rolling(7).mean().plot(color=PL_PINK, lw=3)
plt.title('Tendencia de Goles por Fecha (Media Móvil 7 días)', color=PL_PURPLE, fontsize=14, fontweight='bold')
plt.ylabel('Promedio de Goles por Partido')
plt.savefig('../../img/goal_trend_pl.png')
plt.show()

## 2. Power EDA: Justificación del Factor Humano (Árbitros)
Buscamos si el árbitro influye en el ritmo del juego y en los goles resultantes.

In [ ]:
top_refs = df_matches['referee'].value_counts().head(10).index
df_refs = df_matches[df_matches['referee'].isin(top_refs)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_refs, x='referee', y='total_goals', palette='magma')
plt.xticks(rotation=45)
plt.title('Impacto del Árbitro en el Volumen de Goles (Top 10)', color=PL_PURPLE, fontweight='bold')
plt.savefig('../../img/referee_impact_pl.png')
plt.show()

# Prueba de Kruskal-Wallis
groups = [group['total_goals'].values for name, group in df_refs.groupby('referee')]
stat, p = kruskal(*groups)
print(f"Prueba de Kruskal-Wallis (p-value): {p:.4f}")

## 3. Power EDA: La Hipótesis de la Fatiga
Demostramos visualmente la correlación negativa entre los días de descanso y la probabilidad de un marcador abultado.

In [ ]:
df_golden = pd.read_csv('../../data/matches_golden_features.csv')

plt.figure(figsize=(8, 6))
sns.regplot(data=df_golden, x='Home_Days_Rest', y='total_goals', 
            scatter_kws={'alpha':0.4, 'color':PL_PURPLE}, line_kws={'color':PL_CYAN, 'lw':3})
plt.title('Relación: Días de Descanso vs Goles Totales', color=PL_PURPLE, fontweight='bold')
plt.savefig('../../img/fatigue_correlation_pl.png')
plt.show()

## 4. Modelado y Validación de Residuos
Lasso Regression con log-target y comprobación de supuestos.

In [ ]:
features = ['Home_Avg_FTHG', 'Away_Avg_AST', 'Home_Days_Rest', 'Home_Sum_influence']
X = df_golden[features]
y_log = np.log1p(df_golden['total_goals'])

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LassoCV(cv=5))
])

pipe.fit(X, y_log)

# Residuos
y_pred = pipe.predict(X)
res = y_log - y_pred

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(res, kde=True, ax=ax[0], color=PL_PURPLE)
sm.qqplot(res, line='45', fit=True, ax=ax[1])
plt.savefig('../../img/residuos_final_pl.png')
plt.show()